# Model Selection from Multiple Algorithm
options will be like
  - XGBoost (XGB)
  - LightGBM
  - Random Forest (RF)
  - Support Vector Machine (SVM)
  - K-Nearest Neighbors (KNN)
  - Logistic Regression
  - Naive Bayes

Not trying:
  - Gradient Boosting (Generally, XGB and LightGBM perform better than it)
  - Adabost (Generally, XGB and LightGBM perform better than it)
  - Decision Tree (Generally, RF perform better than it)
  - Catboost (all features are not categorical)

Also perform hyperparameter tunning on the above algorithms using Bayesian Optimization (e.g., Optuna) and log the best result of an algorithm as a run.

NOTE: here, Experiment 5 has been done in different files as Sequentially running of these algorithm takes a lot of time.

**Naive Bayes**

In [1]:
import os
from google.colab import userdata

os.environ["AWS_ACCESS_KEY_ID"] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ["AWS_DEFAULT_REGION"] = userdata.get('AWS_DEFAULT_REGION')
os.environ["satya_mlflow_ec2_uri"] = userdata.get('satya_mlflow_ec2_uri')

In [2]:
!pip install mlflow boto3 awscli optuna imbalanced-learn
!aws sts get-caller-identity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.0/314.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85

In [3]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from imblearn.over_sampling import SMOTE
import mlflow.sklearn
import pandas as pd
import mlflow, optuna


In [4]:
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri(os.environ["satya_mlflow_ec2_uri"])

# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning")

<Experiment: artifact_location='s3://satya-mlflow-bucket/188737626132630835', creation_time=1758703381598, experiment_id='188737626132630835', last_update_time=1758703381598, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [5]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Youtube_Comment_Sentiment_Analysis/reddit_preprocessing.csv').dropna()
df.shape

Mounted at /content/drive


(36662, 2)

In [6]:
# Step 1: (Optional) Remapping - skipped since not strictly needed for Multinomial Naive Bayes

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 1000  # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['clean_comment'])
y = df['category']

# Step 4: Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, name=f"{model_name}_model")


# Step 6: Optuna objective function for Multinomial Naive Bayes
def objective_mnb(trial):
    alpha = trial.suggest_float('alpha', 1e-4, 1.0, log=True)  # Tuning the smoothing parameter

    # MultinomialNB model setup
    model = MultinomialNB(alpha=alpha)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))


# Step 7: Run Optuna for Multinomial Naive Bayes, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_mnb, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = MultinomialNB(alpha=best_params['alpha'])

    # Log the best model with MLflow, passing the algo_name as "MultinomialNB"
    log_mlflow("MultinomialNB", best_model, X_train, X_test, y_train, y_test)

# Run the experiment for Multinomial Naive Bayes
run_optuna_experiment()


[I 2025-09-24 09:31:33,045] A new study created in memory with name: no-name-0c6974b4-1bfb-4e93-a941-05fa83d606c8
[I 2025-09-24 09:31:33,064] Trial 0 finished with value: 0.6684633269921793 and parameters: {'alpha': 0.00239160518597514}. Best is trial 0 with value: 0.6684633269921793.
[I 2025-09-24 09:31:33,082] Trial 1 finished with value: 0.6682519551891778 and parameters: {'alpha': 0.010664805934153691}. Best is trial 0 with value: 0.6684633269921793.
[I 2025-09-24 09:31:33,097] Trial 2 finished with value: 0.6666666666666666 and parameters: {'alpha': 0.1601507352706202}. Best is trial 0 with value: 0.6684633269921793.
[I 2025-09-24 09:31:33,109] Trial 3 finished with value: 0.6659268653561615 and parameters: {'alpha': 0.36266921132317936}. Best is trial 0 with value: 0.6684633269921793.
[I 2025-09-24 09:31:33,121] Trial 4 finished with value: 0.6675121538786726 and parameters: {'alpha': 0.08184025730378688}. Best is trial 0 with value: 0.6684633269921793.
[I 2025-09-24 09:31:33,133

🏃 View run MultinomialNB_SMOTE_TFIDF_Trigrams at: http://65.2.37.109:5000/#/experiments/188737626132630835/runs/dfa0128698ef4bb3986f894dd93dc2aa
🧪 View experiment at: http://65.2.37.109:5000/#/experiments/188737626132630835
